In [124]:
import scipy.optimize as opt
import numpy as np
import pandas as pd
import os
#PyGEM
import pygem.pygem_input as pygem_prms
import pygem.oggm_compat as oggm

In [32]:
# READ GULKANA ELEVATION DATA FROM RGI 6.0
gulkana = '01.00570'
glac_no = [gulkana]

# ----- RGI DATA -----
# Filepath for RGI files
main_directory = os.getcwd()
rgi_fp = main_directory + '/../RGI/rgi60/00_rgi60_attribs/'
assert os.path.exists(rgi_fp), 'RGI filepath does not exist. PyGEM requires RGI data to run.'
# Column names
rgi_lat_colname = 'CenLat'
rgi_lon_colname = 'CenLon_360' # REQUIRED OTHERWISE GLACIERS IN WESTERN HEMISPHERE USE 0 deg
elev_colname = 'elev'
indexname = 'GlacNo'
rgi_O1Id_colname = 'glacno'
rgi_glacno_float_colname = 'RGIId_float'
# Column names from table to drop (list names or accept an empty list)
rgi_cols_drop = ['GLIMSId','BgnDate','EndDate','Status','Linkages','Name']

region = glac_no[0][0:2]
for i in os.listdir(rgi_fp):
    if i.startswith(str(region)) and i.endswith('.csv'):
        rgi_fn = i
glacier_table = pd.read_csv(rgi_fp + rgi_fn,skipinitialspace=True,index_col='RGIId')
glacier_table.drop(rgi_cols_drop, axis=1, inplace=True)
gulkana_tbl = glacier_table.loc['RGI60-'+glac_no[0]]
print('Gulkana Glacier Table:')
print(gulkana_tbl)
elev_points = gulkana_tbl[['Zmin','Zmed','Zmax']]

Gulkana Glacier Table:
CenLon      -145.427
CenLat        63.281
O1Region       1.000
O2Region       2.000
Area          17.567
Zmin        1162.000
Zmax        2438.000
Zmed        1858.000
Slope         14.000
Aspect       172.000
Lmax        8639.000
Connect        0.000
Form           0.000
TermType       0.000
Surging        9.000
Name: RGI60-01.00570, dtype: float64


In [8]:
#%% MODEL PROPERTIES
density_ice = 900           # Density of ice [kg m-3] (or Gt / 1000 km3)
density_water = 1000        # Density of water [kg m-3]
area_ocean = 362.5 * 1e12   # Area of ocean [m2] (Cogley, 2012 from Marzeion et al. 2020)
k_ice = 2.33                # Thermal conductivity of ice [J s-1 K-1 m-1] recall (W = J s-1)
k_air = 0.023               # Thermal conductivity of air [J s-1 K-1 m-1] (Mellor, 1997)
k_air = 0.001               # Thermal conductivity of air [J s-1 K-1 m-1]
ch_ice = 1890000            # Volumetric heat capacity of ice [J K-1 m-3] (density=900, heat_capacity=2100 J K-1 kg-1)
ch_air = 1297               # Volumetric Heat capacity of air [J K-1 m-3] (density=1.29, heat_capacity=1005 J K-1 kg-1)
Lh_rf = 333550              # Latent heat of fusion [J kg-1]
tolerance = 1e-12           # Model tolerance (used to remove low values caused by rounding errors)
gravity = 9.81              # Gravity [m s-2]
pressure_std = 101325       # Standard pressure [Pa]
temp_std = 288.15           # Standard temperature [K]
R_gas = 8.3144598           # Universal gas constant [J mol-1 K-1]
molarmass_air = 0.0289644   # Molar mass of Earth's air [kg mol-1]

In [107]:
#READ GULKANA ELEVATIONS FROM OGGM GDIRS
gdir = oggm.single_flowline_glacier_directory(glacier_str, logging_level='CRITICAL')
fls = oggm.get_glacier_zwh(gdir)
nonzero_bins = np.nonzero(fls['h'].to_numpy())
fls = fls.iloc[nonzero_bins]
z_stats = np.array([np.min(fls['z']),np.median(fls['z']),np.max(fls['z'])])

print('Min, median and max bin elevations from OGGM flowlines:')
print(pd.Series(z_stats,index=['Zmin','Zmed','Zmax']))
print('Min, median and max elevation from RGI')
print(elev_points)

median_index = np.where(fls['z']==z_stats[1])[0][0]
w_stats = np.array([fls['w'][0],fls['w'][median_index],fls['w'][52]])
h_stats = np.array([fls['h'][0],fls['h'][median_index],fls['h'][52]])
geo_index = ['Bottom','Middle','Top']
geometry = pd.DataFrame({'z':z_stats,'w':w_stats,'h':h_stats},index=geo_index)
print(geometry)

Min, median and max bin elevations from OGGM flowlines:
Zmin    1172.809436
Zmed    1628.401286
Zmax    2330.675374
dtype: float64
Min, median and max elevation from RGI
Zmin    1162.0
Zmed    1858.0
Zmax    2438.0
Name: RGI60-01.00570, dtype: float64
                  z            w           h
Bottom  1172.809436   649.658441   60.315266
Middle  1628.401286  1680.294979  136.998843
Top     2330.675374   168.584994  123.288910


In [ ]:
class GCM():
    """
    Global climate model data properties and functions used to automatically retrieve data.
    
    Attributes
    ----------
    name : str
        name of climate dataset.
    scenario : str
        rcp or ssp scenario (example: 'rcp26' or 'ssp585')
    """
    def __init__(self, 
                 name=str(),
                 scenario=str()):
        """
        Add variable name and specific properties associated with each gcm.
        """
        
        if pygem_prms.rgi_lon_colname not in ['CenLon_360']:
            print('\n\nCHECK HOW NEGATIVE LONGITUDES ARE HANDLED!!!!\n\n')
        
        # Source of climate data
        self.name = name
        # Set parameters for ERA5, ERA-Interim, and CMIP5 netcdf files
        if self.name == 'ERA5':
            # Variable names
            self.temp_vn = 't2m'
            self.tempstd_vn = 't2m_std'
            self.prec_vn = 'tp'
            self.elev_vn = 'z'
            self.lat_vn = 'latitude'
            self.lon_vn = 'longitude'
            self.time_vn = 'time'
            self.lr_vn = 'lapserate'
            # Variable filenames
            self.temp_fn = pygem_prms.era5_temp_fn
            self.tempstd_fn = pygem_prms.era5_tempstd_fn
            self.prec_fn = pygem_prms.era5_prec_fn
            self.elev_fn = pygem_prms.era5_elev_fn
            self.lr_fn = pygem_prms.era5_lr_fn
            # Variable filepaths
            self.var_fp = pygem_prms.era5_fp
            self.fx_fp = pygem_prms.era5_fp
            # Extra information
            self.timestep = pygem_prms.timestep
            self.rgi_lat_colname=pygem_prms.rgi_lat_colname
            self.rgi_lon_colname=pygem_prms.rgi_lon_colname

In [129]:
#manually set number of exponentially scaling bins
n_vert_bins = 10

#ENTER BIN LOOP
for point in geo_index:
    #define surface elvation, width and ice thickness (m) of the current bin
    bin_z, bin_w, bin_h = list(geometry.loc[point])
    
    #find constant to scale bins such that max depth is reached in n bins
    c = opt.fsolve(lambda c: bin_h-np.sum(np.exp(np.arange(n_vert_bins)*c)),10)
    bin_hs = np.exp(c*range(1,n_vert_bins))
    

[ 1.37022498  1.87751649  2.57261999  3.52506817  4.83013646  6.61837362
  9.06866085 12.42610561 17.02656029] 59.315266461246665 60.315266461246665
[ 1.53986905  2.37119668  3.65133237  5.62257369  8.65802719 13.33222808
 20.52988534 31.61333495 48.68039595] 135.9988433032514 136.99884330324653
[ 1.5176481   2.30325577  3.49553175  5.30498714  8.05110367 12.21874222
 18.54375097 28.14288851 42.7110014 ] 122.28890954682605 123.28890954682606


In [12]:
import pygem.pygem_input as pygem_prms
import pickle

In [14]:
glacier_str = glac_no[0][1::]
modelprms_fn =  glacier_str + '-modelprms_dict.pkl'
modelprms_fp = pygem_prms.output_filepath + 'calibration/' + glacier_str.split('.')[0].zfill(2) + '/'
modelprms_fullfn = modelprms_fp + modelprms_fn
assert os.path.exists(modelprms_fullfn), glacier_str + ' calibrated parameters do not exist.'            
with open(modelprms_fullfn, 'rb') as f:
    modelprms_dict = pickle.load(f)